# 🌟 FaceFinder AI — Cloud GPU Live Demo
### Academic Project by Daniel Mwalili Mutinda (JKUAT)
**Supervisor:** Dr. Judy Gateri | **BSc Information Technology (2026)**

---

Welcome! This Colab notebook launches the full **FaceFinder AI** system powered by a free **Nvidia T4 Cloud GPU**.

### 🚀 Quick Start (Just 2 clicks!):
1. In the top menu, verify GPU is enabled: **Runtime** -> **Change runtime type** -> Select **T4 GPU** -> **Save**.
2. Click **Runtime** -> **Run all** (or run each cell below in order).
3. At the bottom of Cell 3 or 4, click your generated **Public Link** to open FaceFinder AI on any device worldwide!

In [ ]:
#@title 1. Verify GPU & Clone Repository
!nvidia-smi

import os
if not os.path.exists("facefinder"):
    !git clone https://github.com/Officialkid/facefinder.git
%cd facefinder
!git pull origin main

In [ ]:
#@title 2. Install AI Dependencies & Download Cloudflare Tunnel
%cd /content/facefinder
!pip install -q -r image-sorter/backend/requirements.txt
!pip install -q tf-keras gradio pyngrok

# Install Cloudflare tunnel for instant, zero-token public HTTPS sharing
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("Dependencies and Cloudflare Tunnel installed successfully!")

In [ ]:
#@title 3. Launch FastAPI Backend + Public HTTPS Link
import subprocess
import time
import re
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

# 1. Start FastAPI Backend in background
backend_proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/facefinder/image-sorter/backend",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
print("Starting FastAPI backend on port 8000...")
time.sleep(6)

# 2. Start Cloudflare Tunnel to expose the Backend
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Generating your public shareable link...")
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_proc.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(0.5)

if tunnel_url:
    print("\n" + "="*70)
    print(f"SUCCESS! YOUR LIVE PUBLIC LINK IS READY:")
    print(f"{tunnel_url}/docs")
    print("="*70)
    print(f"Share this URL with your supervisor, examiner, or friend!")
    print(f"API Swagger Documentation & Live Scanning: {tunnel_url}/docs")
    print("="*70)
else:
    print("Backend is running on http://localhost:8000.")


### 💡 Interactive Visual Web UI (Easiest to Share!)
Run the cell below to launch a complete visual Web Interface with image dropzones, camera upload, threshold slider, and instant results gallery with a **72-hour public sharing link**!

In [ ]:
#@title 4. Launch Interactive Web App (Generates https://*.gradio.live link)
import gradio as gr
import cv2
import numpy as np
from pathlib import Path
import tempfile
import sys
sys.path.append("/content/facefinder/image-sorter/backend")
from app.services.face_recognition import extract_embedding, match_face_in_image, warmup_models

warmup_models("ArcFace")

def find_faces(reference_img, dataset_files, threshold):
    if reference_img is None:
        return "Please upload a reference photo (your face).", []
    if not dataset_files:
        return "Please upload event photos to scan.", []
    
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp_ref:
        cv2.imwrite(tmp_ref.name, cv2.cvtColor(reference_img, cv2.COLOR_RGB2BGR))
        ref_emb = extract_embedding(tmp_ref.name, model_name="ArcFace", target_face_index=0)
    
    if ref_emb is None:
        return "No face detected in reference photo. Please try a clearer portrait.", []
    
    matches = []
    for file_obj in dataset_files:
        path = file_obj.name if hasattr(file_obj, "name") else str(file_obj)
        is_match, is_cand, sim, dist, face_cnt, blur = match_face_in_image(
            path, ref_emb, model_name="ArcFace", distance_threshold=threshold
        )
        if is_match or is_cand:
            label = f"Verified: {int(sim*100)}%" if is_match else f"Candidate: {int(sim*100)}%"
            matches.append((path, label))
    
    matches.sort(key=lambda x: x[1], reverse=True)
    return f"Scan Complete! Found {len(matches)} matching photos out of {len(dataset_files)}.", matches

demo = gr.Interface(
    fn=find_faces,
    inputs=[
        gr.Image(label="1. Reference Photo (Your Face)", type="numpy"),
        gr.File(label="2. Event Photos (Upload multiple or ZIP)", file_count="multiple"),
        gr.Slider(0.2, 0.7, value=0.45, step=0.05, label="Similarity Threshold (0.45 recommended)")
    ],
    outputs=[
        gr.Textbox(label="Status & Metrics"),
        gr.Gallery(label="Matching Event Photos", columns=3, height="auto")
    ],
    title="FaceFinder AI — Cloud GPU Photo Retrieval",
    description="AI-powered face recognition that retrieves only photos of you from large event albums. Powered by ArcFace on GPU.",
    theme=gr.themes.Soft(primary_hue="violet", secondary_hue="cyan")
)

demo.launch(share=True, debug=True)
